In [1]:
import os
import pandas as pd
import jupyter_black

from datetime import datetime


jupyter_black.load()

# import openpyxl

Here's what we want out of our mapper.

If we imagine the table has a column for each facility/metric-attribute pair with metatdata attached, we can reconceive all of this for fully automated processing into a BI format.

We need to establish a full list of **context** fields, and we need a list of **attribute** fields that we can automatically populate for the full list of facilities and metrics



In [2]:
context_fields = ["Year", "EntryType", "Scenario"]

attr_fields = [
    "BusinessUnit",
    "Facility",
    "Metric",
    "Unit",
    "Scope",
    "MeasureType",
    "Category",
]

In [3]:
# import pandas as pd

# # Replace 'your_file.xlsx' with the path to your Excel file
# file_path = r"C:\Users\GreenSte\OneDrive - Parametrix, Inc\Projects\Fresherized\Parametrix-Fresherized External\Avomex facilities\Facilities ISO 50001\Equipment Uses Sabinas 2024.xlsx"

# # Read the Excel file into a DataFrame
# df = pd.read_excel(file_path, engine='openpyxl', )

# # Display the first few rows of the DataFrame
# print(df.head())

In [ ]:
filepath = os.path.join(
    r"C:\Users\GreenSte\OneDrive - Parametrix, Inc\2025 Work\Megamex External Sharepoint 2025\Climate Action\MMX BI Dashboard Workspace\inputs\bi_tables_working_draft_9-25-25.xlsx"
)

file = pd.ExcelFile(filepath, engine="openpyxl")

sheet_names = file.sheet_names
print("Sheets in workbook:", sheet_names)

file.close()

In [ ]:
# Includes all signed action deltas - does not include no action baseline
actions = ["refs", "r99", "compressed_air", "screwpress", "waste_to_fuel"]

mappings_camel = [
    "RefsMapping",
    "R99Mapping",
    "CompressorMapping",
    "ScrewpressMapping",
    "WasteToFuelMapping",
]

action_mapping = dict(zip(actions, mappings_camel))

action_mapping

In [ ]:
for i, e in action_mapping.items():
    print(i, e)

In [ ]:
# def map_and_transform_for_bi(
#     action_map=action_mapping,
#     filepath=filepath,
#     usecols="A:C",
#     skiprows=0,
#     exclusions=["Category"],
# ):

#     factoids = []

#     for action_name, mapping_name in action_map.items():
#         # Read in the action and mapping sheets
#         action_wide = pd.read_excel(
#             filepath, sheet_name=action_name, skiprows=skiprows, engine="openpyxl"
#         )

#         mapping = pd.read_excel(
#             filepath,
#             sheet_name=mapping_name,
#             skiprows=0,
#             usecols=usecols,
#             engine="openpyxl",
#         )
#         mapping = mapping.loc[~mapping["TargetField"].isin(exclusions)]

#         # Keep context columns straight from the sheet
#         context_cols = [
#             c for c in ["Year", "EntryType", "Scenario"] if c in action_wide.columns
#         ]
#         value_cols = [c for c in action_wide.columns if c not in context_cols]

#         # Melt the wide data to long with a generic Value column

#         long = action_wide.melt(
#             id_vars=context_cols,
#             value_vars=value_cols,
#             var_name="SourceColumn",
#             value_name="Value",
#         )  # .dropna(subset=["Scenario"])

#         # Join attributes from the mapping table
#         attr = mapping.pivot_table(
#             index="SourceColumn",
#             columns="TargetField",
#             values="TargetValue",
#             aggfunc="first",
#         ).reset_index()

#         long = long.merge(attr, on="SourceColumn", how="left")

#         # Filter to BAU / No Action
#         if "Scenario" in long.columns:
#             long = long[
#                 long["Scenario"].astype(str).str.strip().str.lower().eq("baseline")
#             ]

#         # Split into Activity vs Emissions
#         long["MeasureType"] = long["MeasureType"].fillna("Activity")
#         factoid = long.copy()

#         factoid["VersionStamp"] = datetime.now()
#         factoids.append(factoid)

#         return factoids

# mappings = []
# for mapping_name in mapping_sheets:
#     sheet_wide = pd.read_excel(filepath, sheet_name=sheet_name, skiprows=skiprows, engine='openpyxl')

In [ ]:
# action_fact = pd.concat(map_and_transform_for_bi())
# action_fact

In [ ]:
no_action_wide = pd.read_excel(
    filepath, sheet_name="baseline", skiprows=2, engine="openpyxl"
)
# no_action_wide.columns = [str.lower(i) for i in no_action_wide.columns]

no_action_mapping = pd.read_excel(
    filepath, sheet_name="NoActionMapping", usecols="A:C", engine="openpyxl"
)
# no_action_mapping.columns = [str.lower(i) for i in no_action_mapping.columns]

no_action_mapping = no_action_mapping.loc[
    no_action_mapping["TargetField"] != "Category"
]

In [ ]:
# Keep context columns straight from the sheet
context_cols = [
    c for c in ["Year", "EntryType", "Scenario"] if c in no_action_wide.columns
]
value_cols = [c for c in no_action_wide.columns if c not in context_cols]

In [ ]:
# Melt the wide data to long with a generic Value column
long = no_action_wide.melt(
    id_vars=context_cols,
    value_vars=value_cols,
    var_name="SourceColumn",
    value_name="Value",
).dropna(subset=["Scenario"])

long

In [ ]:
# Join attributes from the mapping table
attr = no_action_mapping.pivot_table(
    index="SourceColumn", columns="TargetField", values="TargetValue", aggfunc="first"
).reset_index()

# attr.SourceColumn = attr["SourceColumn"].str.lower()

# attr = attr.loc[attr[""]]
attr

In [ ]:
long = long.merge(attr, on="SourceColumn", how="left")

In [ ]:
long

In [ ]:
# Filter to BAU / No Action
if "Scenario" in long.columns:
    long = long[long["Scenario"].astype(str).str.strip().str.lower().eq("baseline")]


# Split into Activity vs Emissions
long["MeasureType"] = long["MeasureType"].fillna("Activity")
fact = long.copy()

fact["VersionStamp"] = datetime.now()

In [ ]:
fact

In [ ]:
# fact["MonthKey"] = (
#     fact["Year"].astype(int) * 100 + 1
# )  # if you only have Year; adjust if Month exists


# fact["ActivityValue"] = fact.apply(
#     lambda r: r["Value"] if r["MeasureType"].lower() == "activity" else None, axis=1
# )
# fact["Emissions_tCO2e"] = fact.apply(
#     lambda r: r["Value"] if r["MeasureType"].lower() == "emissions" else None, axis=1
# )

# # Optional: tag Actual/Forecast simply by year cutover (replace with your rule if you have one)
# cutover_year = 2025
# fact["DataType"] = fact["Year"].apply(
#     lambda y: "Actual" if int(y) <= cutover_year - 1 else "Forecast"
# )


# fact.rename(columns={"Calendar Year":"Year"}, inplace=True)
# fact.rename(columns={"Action":"ActionID"}, inplace=True)

# Inventory table has been built. Now let's run the action deltas. We will export them both at the end.

In [ ]:
actions
mappings_camel

In [ ]:
action_dfs = []

for action in actions:
    act_df = pd.read_excel(filepath, engine="openpyxl", sheet_name=action)
    act_df = act_df.dropna()
    action_dfs.append(act_df)

mapping_dfs = []

for mapping in mappings_camel:
    map_df = pd.read_excel(
        filepath, engine="openpyxl", sheet_name=mapping, usecols="A:C"
    )
    map_df = map_df.loc[map_df["TargetField"] != "Category"]
    map_df = map_df.dropna()
    mapping_dfs.append(map_df)

In [ ]:
# # Keep context columns straight from the sheet
# context_cols_refs = (
#     mapping_dfs[0]
#     .loc[mapping_dfs[0]["TargetField"] == "Context"]["SourceColumn"]
#     .values
# ).tolist()
# value_cols_refs = [c for c in action_dfs[0].columns if c not in context_cols_refs]

# context_cols_r99 = (
#     mapping_dfs[1]
#     .loc[mapping_dfs[1]["TargetField"] == "Context"]["SourceColumn"]
#     .values
# ).tolist()
# value_cols_r99 = [c for c in action_dfs[1].columns if c not in context_cols_r99]

In [ ]:
actions_list = []

# Iterate over actions and maps and process for loading
for df, mapping in list(zip(action_dfs, mapping_dfs)):
    context_cols = mapping.loc[mapping["TargetField"] == "Context"][
        "SourceColumn"
    ].values
    value_cols = [c for c in df.columns if c not in context_cols]
    print(context_cols, value_cols)

    melted_df = df.melt(
        id_vars=context_cols,
        value_vars=value_cols,
        var_name="SourceColumn",
        value_name="Value",
    )

    # Join attributes from the mapping table
    attr = mapping.pivot_table(
        index="SourceColumn",
        columns="TargetField",
        values="TargetValue",
        aggfunc="first",
    ).reset_index()

    melted_df = melted_df.merge(attr, on="SourceColumn", how="left")

    melted_df = melted_df.dropna(subset="MeasureType")

    actions_list.append(melted_df)


# Concatenate to a deltas table
action_deltas = pd.concat(actions_list)
action_deltas["VersionStamp"] = datetime.now()

In [ ]:
actions_list[-1]

In [ ]:
# Create UI-friendly mapping values
display_mapping = {
    "Action": {
        "refrigerant_transition": "Refrigerant Transition",
        "r99_substitution": "R99 Substitution",
        "compressed_air": "Compressed Air",
        "waste_to_fuel": "Waste to Fuel",
        "screwpress": "Screwpress",
    },
    "MeasureType": {"activity": "Activity", "emissions": "Emissions"},
    "Metric": {
        "refrigerants": "Refrigerants",
        "electricity_renewable": "Renewable Electricity",
        "diesel": "Diesel",
        "gasoline": "Gasoline",
        "natural_gas": "Natural Gas",
        "propane": "Propane",
        "steam": "Steam",
        "water": "Water",
        "electricity_grid": "Grid Electricity",
        "electricity": "Electricity",
        "scope3_upstreamfuels": "Scope 3: Upstream Fuels",
        "waste": "Waste",
        "commute": "Commute",
        "scope3_transport": "Scope 3: Transport",
        "revenue": "Revenue",
        "production": "Production",
        "ethanol": "Ethanol",
        "biodiesel": "Biodiesel",
        "renewable_diesel": "Renewable Diesel",
        "supply chain": "Supply Chain",
    },
    "Scope": {
        "scope 1": "Scope 1",
        "scope 2": "Scope 2",
        "scope 3": "Scope 3",
        "n/a": "N/A",
    },
    "Unit": {
        "usd - nominal": "USD",
        "mtco2e": "Metric Tons CO2e",
        "kwh": "kWh",
        "therm": "Therms",
        "gal": "Gallons",
        "lbs": "Pounds",
        "ton": "Short Tons",
        "cubic feet": "Cubic Feet",
        "cubic yards": "Cubic Yards",
        "mmbtu": "MMBtu",
        "mcf": "MCF",
        "percentage": "Percentage",
        "passenger miles": "Passenger Miles",
    },
}

for key in display_mapping.keys():
    if key != "Action":
        fact[f"{key}_display"] = fact[key].map(display_mapping[key])

    action_deltas[f"{key}_display"] = action_deltas[key].map(display_mapping[key])

In [ ]:
# Final selection for Power BI FactInventory (BAU)
fact_out_cols = [
    "Scenario",
    "Year",
    "EntryType",
    "Facility",
    "Metric_display",
    "Unit_display",
    "Scope_display",
    "MeasureType_display",
    "Value",
    "SourceColumn",
    "VersionStamp",
]


fact[fact_out_cols].to_csv(
    r"C:\Users\GreenSte\OneDrive - Parametrix, Inc\2025 Work\Megamex External Sharepoint 2025\Climate Action\MMX BI Dashboard Workspace\py_out\InventoryFact.csv",
    index=False,
)
print("Wrote InventoryFact.csv")

In [ ]:
action_deltas

In [ ]:
# Final selection for Power BI FactInventory (BAU)
delta_out_cols = [
    "Action_display",
    "Year",
    "EntryType",
    "Facility",
    "Metric_display",
    "Unit_display",
    "Scope_display",
    "MeasureType_display",
    "Value",
    "SourceColumn",
    "VersionStamp",
]


action_deltas[delta_out_cols].to_csv(
    r"C:\Users\GreenSte\OneDrive - Parametrix, Inc\2025 Work\Megamex External Sharepoint 2025\Climate Action\MMX BI Dashboard Workspace\py_out\ActionDeltaFact.csv",
    index=False,
)
print("Wrote ActionDeltaFact.csv")

# END OF CODE